In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-262kvjog
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-262kvjog
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=41e69aa7ea21c9dbe0d8dc8aef1d03aae6bb8ba19a4c4b9468bf61652464cf75
  Stored in directory: /tmp/pip-ephem-wheel-cache-1dt67tac/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE_URL = "/content/drive/MyDrive/RL Project"


In [4]:
!pip install matplotlib torch numpy Pillow

In [ ]:
#!/usr/bin/env python3
"""
RL Text-to-Drawing Agent (Proof of Concept)
"""

import argparse
import os
import time
from collections import deque

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from torch.distributions import Normal
from PIL import Image

try:
    import clip
except ImportError:
    raise ImportError(
        "Install CLIP:  pip install git+https://github.com/openai/CLIP.git"
    )


# ══════════════════════════════════════════════════════════════════════════════
#  Hyper-parameters
# ══════════════════════════════════════════════════════════════════════════════
RANDOM_SEED = 469

CANVAS_SIZE = 128
RENDERER_PARAMS = 10
STROKE_COLOR = 3
STROKE_PARAMS = RENDERER_PARAMS + STROKE_COLOR  # 13

STROKES_PER_STEP = 5
N_STROKE_BUNDLES = 5  # steps (number of bundles) per episode

CLIP_EMBED_DIM = 512
CNN_DIM = 256
SKETCH_DIM = 512  # CLIP visual embed of reference sketch
STEP_DIM = 1  # normalised step counter
STATE_DIM = (
    CLIP_EMBED_DIM  # clip canvas
    + CLIP_EMBED_DIM  # clip text
    + CNN_DIM  # cnn canvas
    + SKETCH_DIM  # clip sketch
    + STEP_DIM
)  # 1793

HIDDEN_DIM = 512
ACTION_DIM = STROKES_PER_STEP * STROKE_PARAMS  # 65

# ── Per-stroke parameter slices (within a 13-d stroke vector) ──────────────
# Stroke params: (x0, y0, x1, y1, x2, y2, r0, t0, r1, t1, R, G, B)
#   (x0,y0), (x1,y1), (x2,y2) — QBC control points
#   r0, t0 — radius and transparency at the start endpoint
#   r1, t1 — radius and transparency at the end   endpoint
#   R, G, B — stroke colour
_POS_SLICE   = slice(0, 6)   # x0,y0,x1,y1,x2,y2
_R_SLICE     = slice(6, 8)   # r0, t0  (start: radius, transparency)
_RT_SLICE    = slice(8, 10)  # r1, t1  (end:   radius, transparency)
_COLOR_SLICE = slice(10, 13) # R, G, B

# ── Actor mean bias per parameter type ────────────────────────────────────
_BIAS_POS   = 0.0
_BIAS_R     = 0.0  # stroke radius bias - helps combat blob issue
_BIAS_T     = 0.0
_BIAS_COLOR = 0.0

# --- Reward ---
REWARD_SCALE    = 100.0
REWARD_CLIP     = 200.0  # the reward ceiling
TEXT_SIM_WEIGHT   = 1.0
SKETCH_SIM_WEIGHT = 0.0
DIV_BONUS_W       = 0.1

# Stroke-size penalty grows linearly from MIN (step 0) → MAX (last step).
# Low early → agent free to paint big shapes; high late → forced to use fine strokes.
STROKE_SIZE_PENALTY_MIN = 0.1
STROKE_SIZE_PENALTY_MAX = 0.1

BINARY_PENALTY_MASK = False
TERMINAL_ABS_W      = 100

# --- Center reward ---
# Gaussian mask centred on the canvas. Rewards strokes landing near the middle.
CENTER_REWARD_ENABLED = False   # ← flip this to turn it on/off
CENTER_REWARD_W       = 60.0   # reward scale
CENTER_REWARD_SIGMA   = 0.20   # Gaussian std as a fraction of canvas size

# --- Model-based gradient ---
MODEL_BASED_W = 0.05
MB_SAMPLES    = 16

# --- PPO ---
LR            = 5e-4
LR_MIN        = 2e-6
GAMMA         = 0.99
LAMBDA_GAE    = 0.98
CLIP_EPS      = 0.15
ENTROPY_START = 0.02
ENTROPY_END   = 0.002
VALUE_COEF    = 0.5
PPO_EPOCHS    = 4
MINI_BATCH    = 64
UPDATE_EVERY  = 20 * N_STROKE_BUNDLES

# --- Logging ---
LOG_EVERY = 20
PATIENCE  = 3000

# CLIP normalisation constants
_CLIP_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073])
_CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711])

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ══════════════════════════════════════════════════════════════════════════════
#  Neural Renderer  (frozen)
# ══════════════════════════════════════════════════════════════════════════════


class FCN(nn.Module):
    def __init__(self):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(10, 512)
        self.fc2 = nn.Linear(512, 1024)
        self.fc3 = nn.Linear(1024, 2048)
        self.fc4 = nn.Linear(2048, 4096)
        self.conv1 = nn.Conv2d(16, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(8, 16, 3, 1, 1)
        self.conv4 = nn.Conv2d(16, 16, 3, 1, 1)
        self.conv5 = nn.Conv2d(4, 8, 3, 1, 1)
        self.conv6 = nn.Conv2d(8, 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = x.view(-1, 16, 16, 16)
        x = F.relu(self.conv1(x))
        x = self.pixel_shuffle(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pixel_shuffle(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.pixel_shuffle(self.conv6(x))
        x = torch.sigmoid(x)
        return 1 - x.view(-1, 128, 128)  # raw output: 0=stroke, 1=background


class NeuralRenderer(nn.Module):

    def __init__(self, path: str, device: torch.device):
        super().__init__()
        self.net = FCN()
        state_dict = torch.load(path, map_location=device, weights_only=True)
        self.net.load_state_dict(state_dict)
        self.net = self.net.to(device)
        self.net.eval()
        for p in self.net.parameters():
            p.requires_grad_(False)

        with torch.no_grad():
            dummy = torch.zeros(1, RENDERER_PARAMS, device=device)
            out = self.forward(dummy)
        assert out.dim() == 3, f"Unexpected renderer output shape: {out.shape}"
        self.out_h, self.out_w = out.shape[1], out.shape[2]
        print(
            f"[NeuralRenderer] '{path}'  "
            f"in=({RENDERER_PARAMS},) → alpha=({self.out_h}×{self.out_w})"
        )

    def forward(self, params: torch.Tensor) -> torch.Tensor:
        out = self.net(params)  # raw FCN: 0=stroke, 1=background
        if out.dim() == 4:
            out = out[:, 0]
        out = 1.0 - out  # match ddpg.py inversion: 1=stroke, 0=background
        return out.clamp(0.0, 1.0)


# ══════════════════════════════════════════════════════════════════════════════
#  Differentiable canvas operations
# ══════════════════════════════════════════════════════════════════════════════


def _composite(canvas: torch.Tensor, alpha: torch.Tensor, color: torch.Tensor):
    """
    canvas (3,H,W), alpha (H,W) in [0,1], color (3,) → (3,H,W)

    alpha == 1 → paint foreground colour (stroke region)
    alpha == 0 → preserve canvas        (background)

    Matches ddpg.py:
        canvas = canvas * (1 - stroke) + color_stroke
    where color_stroke = stroke * color
    """
    fore = color.view(3, 1, 1).expand_as(canvas)
    return (canvas * (1.0 - alpha.unsqueeze(0)) + fore * alpha.unsqueeze(0)).clamp(
        0.0, 1.0
    )


def _resize_alpha(alpha: torch.Tensor, H: int, W: int) -> torch.Tensor:
    if alpha.shape[-2:] == torch.Size([H, W]):
        return alpha
    return F.interpolate(
        alpha.unsqueeze(1), size=(H, W), mode="bilinear", align_corners=False
    ).squeeze(1)


def render_bundle(
    canvas: torch.Tensor,
    action_01: torch.Tensor,
    renderer: NeuralRenderer,
) -> tuple:
    """
    Render STROKES_PER_STEP strokes sequentially. Differentiable w.r.t. action_01.

    canvas    : (3, H, W) in [0, 1]
    action_01 : (ACTION_DIM,) in [0, 1]

    Returns: (new_canvas (3,H,W), mean_alpha float)
        mean_alpha = average stroke coverage across all strokes.
    """
    H, W = canvas.shape[1], canvas.shape[2]
    bundle = action_01.view(STROKES_PER_STEP, STROKE_PARAMS)
    total_alpha = 0.0

    for i in range(STROKES_PER_STEP):
        geo   = bundle[i, :RENDERER_PARAMS]
        color = bundle[i, RENDERER_PARAMS:]

        alpha = renderer(geo.unsqueeze(0)).squeeze(0)
        alpha = _resize_alpha(alpha.unsqueeze(0), H, W).squeeze(0)

        if BINARY_PENALTY_MASK:
            binary_mask  = (alpha.detach() > 0.0).float()
            total_alpha += binary_mask.mean().item()
        else:
            total_alpha += alpha.detach().mean().item()

        canvas = _composite(canvas, alpha, color)

    mean_alpha = total_alpha / STROKES_PER_STEP
    return canvas, mean_alpha


def render_bundle_batched(
    canvases: torch.Tensor,
    action_bundles: torch.Tensor,
    renderer: NeuralRenderer,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Batched render for model-based loss.
    Returns: (new_canvases, mean_alpha)

    Compositing formula mirrors ddpg.py decode (vectorised):
        canvas = canvas * (1 - stroke) + stroke * color
    """
    B, _, H, W = canvases.shape
    bundles     = action_bundles.view(B, STROKES_PER_STEP, STROKE_PARAMS)
    total_alpha = torch.zeros(B, device=canvases.device)

    for i in range(STROKES_PER_STEP):
        geos   = bundles[:, i, :RENDERER_PARAMS]
        colors = bundles[:, i, RENDERER_PARAMS:]

        alphas = renderer(geos)
        alphas = _resize_alpha(alphas, H, W)

        if BINARY_PENALTY_MASK:
            binary_mask  = (alphas.view(B, -1) > 0.0).float()
            total_alpha += binary_mask.mean(dim=1)
        else:
            total_alpha += alphas.view(B, -1).mean(dim=1)

        fore     = colors.view(B, 3, 1, 1).expand(B, 3, H, W)
        canvases = (
            canvases * (1.0 - alphas.unsqueeze(1)) + fore * alphas.unsqueeze(1)
        ).clamp(0.0, 1.0)

    mean_alpha = total_alpha / STROKES_PER_STEP
    return canvases, mean_alpha


# ══════════════════════════════════════════════════════════════════════════════
#  CLIP helpers
# ══════════════════════════════════════════════════════════════════════════════


def clip_encode_image(img_tensor: torch.Tensor, clip_model, device) -> torch.Tensor:
    img  = F.interpolate(
        img_tensor.unsqueeze(0).float(),
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
    )
    mean = _CLIP_MEAN.view(1, 3, 1, 1).to(device=device, dtype=img.dtype)
    std  = _CLIP_STD.view(1, 3, 1, 1).to(device=device, dtype=img.dtype)
    img  = (img - mean) / std
    emb  = clip_model.encode_image(img).float()
    return emb / emb.norm(dim=-1, keepdim=True)


def clip_encode_batch(canvases: torch.Tensor, clip_model, device) -> torch.Tensor:
    imgs = F.interpolate(
        canvases.float(), size=(224, 224), mode="bilinear", align_corners=False
    )
    mean = _CLIP_MEAN.view(1, 3, 1, 1).to(device=device, dtype=imgs.dtype)
    std  = _CLIP_STD.view(1, 3, 1, 1).to(device=device, dtype=imgs.dtype)
    imgs = (imgs - mean) / std
    embs = clip_model.encode_image(imgs).float()
    return embs / embs.norm(dim=-1, keepdim=True)


def load_sketch_embed(sketch_path: str, clip_model, device) -> torch.Tensor:
    if sketch_path is None or not os.path.isfile(sketch_path):
        print("[sketch] No sketch provided — sketch embedding zeroed.")
        return torch.zeros(1, SKETCH_DIM, device=device)
    img = (
        Image.open(sketch_path)
        .convert("RGB")
        .resize((CANVAS_SIZE, CANVAS_SIZE), Image.BILINEAR)
    )
    img_t = torch.tensor(
        np.array(img, dtype=np.float32) / 255.0, device=device
    ).permute(2, 0, 1)
    with torch.no_grad():
        emb = clip_encode_image(img_t, clip_model, device)
    print(f"[sketch] Encoded '{sketch_path}' → (1, {SKETCH_DIM})")
    return emb


# ══════════════════════════════════════════════════════════════════════════════
#  Center mask helper
# ══════════════════════════════════════════════════════════════════════════════


def _make_center_mask(H: int, W: int, sigma_frac: float, device) -> torch.Tensor:
    """
    Normalised 2-D Gaussian centred on the canvas.
    sigma_frac : std as a fraction of min(H, W).
    Sums to 1.0 so the reward is a weighted-average paint density.
    """
    cy, cx  = H / 2.0, W / 2.0
    ys      = torch.arange(H, device=device).float()
    xs      = torch.arange(W, device=device).float()
    yy, xx  = torch.meshgrid(ys, xs, indexing="ij")
    sigma   = sigma_frac * min(H, W)
    mask    = torch.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2.0 * sigma ** 2))
    return mask / mask.sum()


# ══════════════════════════════════════════════════════════════════════════════
#  Canvas CNN Encoder
# ══════════════════════════════════════════════════════════════════════════════


class CanvasCNN(nn.Module):
    def __init__(self, out_dim: int = CNN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2),
            nn.GroupNorm(8, 32),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.GroupNorm(16, 128),
            nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.GroupNorm(32, 256),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.proj = nn.Linear(256, out_dim)

    def forward(self, canvas: torch.Tensor) -> torch.Tensor:
        return self.proj(self.net(canvas))


# ══════════════════════════════════════════════════════════════════════════════
#  Policy Network
# ══════════════════════════════════════════════════════════════════════════════


def _make_actor_bias() -> torch.Tensor:
    """
    Initialise the actor output bias to keep strokes thin at the start.

    Stroke params: (x0, y0, x1, y1, x2, y2, r0, t0, r1, t1, R, G, B)

    Radii r0 (idx 6) and r1 (idx 8) are set to _BIAS_R = -3.0
        → sigmoid(-3) ≈ 0.047, so strokes start thin at BOTH endpoints.
    All other params left at 0 (neutral sigmoid(0) = 0.5).
    """
    s    = torch.zeros(STROKE_PARAMS)
    s[6] = _BIAS_R  # r0 — start endpoint radius
    s[8] = _BIAS_R  # r1 — end   endpoint radius
    return s.repeat(STROKES_PER_STEP)


class ActorCritic(nn.Module):
    """
    Four-stream policy network.

    State vector layout (total: 1793-d):
      [0:512]     clip_canvas_embed
      [512:1024]  clip_text_embed
      [1024:1280] zeros (CNN features computed fresh from canvas tensor)
      [1280:1792] clip_sketch_embed
      [1792]      step_norm  ∈ [0, 1]
    """

    def __init__(self):
        super().__init__()

        self.canvas_stream = nn.Sequential(
            nn.Linear(CLIP_EMBED_DIM, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
            nn.GELU(),
        )
        self.text_stream = nn.Sequential(
            nn.Linear(CLIP_EMBED_DIM, HIDDEN_DIM // 2),
            nn.LayerNorm(HIDDEN_DIM // 2),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM // 2, HIDDEN_DIM // 2),
            nn.GELU(),
        )
        self.canvas_cnn  = CanvasCNN(out_dim=CNN_DIM)
        self.sketch_proj = nn.Sequential(nn.Linear(SKETCH_DIM, 128), nn.GELU())
        self.step_proj   = nn.Sequential(nn.Linear(STEP_DIM, 32),    nn.GELU())

        fused_dim = HIDDEN_DIM // 2 + HIDDEN_DIM // 2 + CNN_DIM + 128 + 32  # 928
        self.trunk = nn.Sequential(
            nn.Linear(fused_dim, HIDDEN_DIM),
            nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.GELU(),
        )

        self.actor_mean    = nn.Linear(HIDDEN_DIM, ACTION_DIM)
        self.actor_log_std = nn.Parameter(torch.full((ACTION_DIM,), -1.0))
        self.critic        = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )

        nn.init.orthogonal_(self.actor_mean.weight, gain=0.01)
        with torch.no_grad():
            self.actor_mean.bias.copy_(_make_actor_bias())

    def _encode(self, clip_canvas, clip_text, canvas_px, clip_sketch, step_norm):
        c  = self.canvas_stream(clip_canvas)
        t  = self.text_stream(clip_text)
        v  = self.canvas_cnn(canvas_px)
        sk = self.sketch_proj(clip_sketch)
        st = self.step_proj(step_norm)
        return self.trunk(torch.cat([c, t, v, sk, st], dim=-1))

    def _unpack_state(self, state, canvas):
        clip_canvas  = state[:, :512]
        clip_text    = state[:, 512:1024]
        clip_sketch  = state[:, 1280:1792]
        step_norm    = state[:, 1792:1793]
        return clip_canvas, clip_text, canvas, clip_sketch, step_norm

    def forward(self, state, canvas):
        clip_canvas, clip_text, canvas_px, clip_sketch, step_norm = self._unpack_state(
            state, canvas
        )
        h    = self._encode(clip_canvas, clip_text, canvas_px, clip_sketch, step_norm)
        mean = self.actor_mean(h)
        std  = self.actor_log_std.exp().clamp(1e-4, 1.5).expand_as(mean)
        val  = self.critic(h).squeeze(-1)
        return mean, std, val

    @torch.no_grad()
    def act(self, state, canvas):
        mean, std, val = self.forward(state.unsqueeze(0), canvas.unsqueeze(0))
        dist  = Normal(mean, std)
        raw   = dist.rsample()
        log_p = dist.log_prob(raw).sum(-1)
        return raw.squeeze(0).cpu().numpy(), log_p.squeeze(0), val.squeeze(0)

    def evaluate(self, states, canvases, actions):
        mean, std, val = self.forward(states, canvases)
        dist    = Normal(mean, std)
        log_p   = dist.log_prob(actions).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_p, val, entropy


# ══════════════════════════════════════════════════════════════════════════════
#  Rollout Buffer
# ══════════════════════════════════════════════════════════════════════════════


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def push(self, state, canvas_before_cpu, act, rew, lp, val, done):
        self._states.append(state)
        self._canvases.append(canvas_before_cpu)
        self._acts.append(torch.tensor(act, dtype=torch.float32))
        self._rews.append(float(rew))
        self._lps.append(lp.detach())
        self._vals.append(val.detach())
        self._dones.append(bool(done))

    def __len__(self):
        return len(self._rews)

    def clear(self):
        self._states   = []
        self._canvases = []
        self._acts     = []
        self._rews     = []
        self._lps      = []
        self._vals     = []
        self._dones    = []


# ══════════════════════════════════════════════════════════════════════════════
#  Environment
# ══════════════════════════════════════════════════════════════════════════════


class PaintEnv:

    def __init__(self, text_embed, sketch_embed, clip_model, renderer, device):
        self.text_embed   = text_embed
        self.sketch_embed = sketch_embed
        self.clip_model   = clip_model
        self.renderer     = renderer
        self.device       = device
        self.canvas       = None
        self.episode_colors = []
        self.step_count   = 0
        self.prev_sim     = 0.0

        # Precomputed once; only used when CENTER_REWARD_ENABLED is True.
        self.center_mask = _make_center_mask(
            CANVAS_SIZE, CANVAS_SIZE, CENTER_REWARD_SIGMA, device
        )

        self.reset()

    def _encode_canvas(self):
        with torch.no_grad():
            return clip_encode_image(self.canvas, self.clip_model, self.device)

    def _diversity_bonus(self) -> float:
        if len(self.episode_colors) < 2:
            return 0.0
        cols  = np.array(self.episode_colors, dtype=np.float32)
        diffs = cols[:, None, :] - cols[None, :, :]
        n     = len(cols)
        return float(
            np.linalg.norm(diffs, axis=-1)[np.triu_indices(n, k=1)].mean() / 1.732
        )

    def _build_state(self, canvas_embed):
        step_norm = torch.tensor(
            [self.step_count / N_STROKE_BUNDLES],
            dtype=torch.float32,
            device=self.device,
        )
        return torch.cat(
            [
                canvas_embed.squeeze(0),                    # 512
                self.text_embed.squeeze(0),                 # 512
                torch.zeros(CNN_DIM, device=self.device),   # 256 placeholder
                self.sketch_embed.squeeze(0),               # 512
                step_norm,                                  # 1
            ],
            dim=0,
        )

    def reset(self):
        self.canvas = torch.ones(
            3, CANVAS_SIZE, CANVAS_SIZE, dtype=torch.float32, device=self.device
        )
        self.step_count     = 0
        self.episode_colors = []
        emb                 = self._encode_canvas()
        self.prev_sim       = (emb * self.text_embed).sum().item()
        return self._build_state(emb), self.canvas.detach()

    def get_canvas_snapshot(self):
        return self.canvas.detach().cpu().half()

    def step(self, action: np.ndarray):
        action_t  = torch.tensor(action, dtype=torch.float32, device=self.device)
        action_01 = torch.sigmoid(action_t)

        canvas_before = self.canvas  # save for center-reward delta

        with torch.no_grad():
            new_canvas, mean_alpha = render_bundle(
                self.canvas, action_01, self.renderer
            )
        self.canvas = new_canvas.detach()

        # ── growing stroke-size penalty ──────────────────────────────────────
        # step_count is still the 0-based index of THIS step before incrementing.
        # step_frac goes 0.0 → 1.0 across the episode.
        step_frac      = self.step_count / max(N_STROKE_BUNDLES - 1, 1)
        penalty_w      = STROKE_SIZE_PENALTY_MIN + step_frac * (
            STROKE_SIZE_PENALTY_MAX - STROKE_SIZE_PENALTY_MIN
        )
        stroke_penalty = -penalty_w * mean_alpha

        # ── optional center reward ───────────────────────────────────────────
        center_reward = 0.0
        if CENTER_REWARD_ENABLED:
            # paint_delta: how much each pixel darkened this step, averaged over RGB.
            # Clamp to ignore strokes that lighten (e.g. white strokes on white canvas).
            paint_delta   = (canvas_before - self.canvas).clamp(min=0.0).mean(dim=0)
            center_reward = CENTER_REWARD_W * (paint_delta * self.center_mask).sum().item()

        bundle = action_01.view(STROKES_PER_STEP, STROKE_PARAMS).detach().cpu()
        for i in range(STROKES_PER_STEP):
            self.episode_colors.append(tuple(bundle[i, RENDERER_PARAMS:].tolist()))

        emb      = self._encode_canvas()
        text_sim = (emb * self.text_embed).sum().item()

        if self.sketch_embed.abs().sum() > 1e-6:
            sketch_sim = (emb * self.sketch_embed).sum().item()
            new_sim    = text_sim * TEXT_SIM_WEIGHT + sketch_sim * SKETCH_SIM_WEIGHT
        else:
            new_sim = text_sim

        delta_sim = float(
            np.clip((new_sim - self.prev_sim) * REWARD_SCALE, -REWARD_CLIP, REWARD_CLIP)
        )
        self.prev_sim = new_sim

        self.step_count   += 1
        done               = self.step_count >= N_STROKE_BUNDLES
        div_bonus          = DIV_BONUS_W * self._diversity_bonus() if done else 0.0
        terminal_bonus     = TERMINAL_ABS_W * (text_sim ** 2)      if done else 0.0

        reward = delta_sim + stroke_penalty + center_reward + div_bonus + terminal_bonus
        state  = self._build_state(emb)

        coverage = float((self.canvas.detach() < 0.95).any(dim=0).float().mean())
        return (
            state,
            self.canvas.detach(),
            reward,
            done,
            {
                "clip_sim":   new_sim,
                "coverage":   coverage,
                "mean_alpha": mean_alpha,
                "penalty_w":  penalty_w,      # handy for logging
                "center_rew": center_reward,
            },
        )


# ══════════════════════════════════════════════════════════════════════════════
#  GAE
# ══════════════════════════════════════════════════════════════════════════════


def compute_gae(rewards, values, dones, last_value):
    """
    Computes the Advantage A(s, a) via Generalised Advantage Estimation.
    """
    device  = last_value.device
    dtype   = last_value.dtype
    rewards = torch.tensor(rewards, dtype=dtype, device=device)
    dones   = torch.tensor(dones,   dtype=dtype, device=device)
    values  = torch.stack([v.to(device=device, dtype=dtype) for v in values])
    n       = len(rewards)
    advs    = torch.zeros(n, dtype=dtype, device=device)
    gae     = torch.zeros(1, dtype=dtype, device=device)
    prev    = last_value
    for t in reversed(range(n)):
        mask    = 1.0 - dones[t]
        delta   = rewards[t] + GAMMA * prev * mask - values[t]
        gae     = delta + GAMMA * LAMBDA_GAE * mask * gae
        advs[t] = gae
        prev    = values[t]
    return advs, advs + values


# ══════════════════════════════════════════════════════════════════════════════
#  Model-Based Auxiliary Loss
# ══════════════════════════════════════════════════════════════════════════════


def _model_based_loss(
    policy,
    states_t,
    canvases_cpu,
    mb_indices,
    device,
    renderer,
    text_embed,
    sketch_embed,
    clip_model,
):
    sub_idx = mb_indices[:MB_SAMPLES]
    n       = len(sub_idx)
    if n == 0:
        return torch.zeros([], device=device)

    canvases = torch.stack(
        [canvases_cpu[i.item()].to(device=device, dtype=torch.float32) for i in sub_idx]
    )

    mean, _, _  = policy.forward(states_t[sub_idx], canvases)
    action_01   = torch.sigmoid(mean)

    new_canvases, mean_alpha = render_bundle_batched(canvases, action_01, renderer)
    embs      = clip_encode_batch(new_canvases, clip_model, device)
    text_sims = (embs * text_embed).sum(dim=-1)

    if sketch_embed.abs().sum() > 1e-6:
        sketch_sims = (embs * sketch_embed).sum(dim=-1)
        sims        = (text_sims + sketch_sims) / 2.0
    else:
        sims = text_sims

    # Use max penalty weight as a conservative bound for the model-based loss,
    # consistent with the late-episode behaviour the agent is being trained toward.
    mb_penalty_weight = STROKE_SIZE_PENALTY_MAX / REWARD_SCALE
    return -sims.mean() + (mb_penalty_weight * mean_alpha.mean())


# ══════════════════════════════════════════════════════════════════════════════
#  PPO Update
# ══════════════════════════════════════════════════════════════════════════════


def ppo_update(
    policy,
    optimizer,
    buffer,
    last_val,
    device,
    entropy_coef,
    renderer,
    text_embed,
    sketch_embed,
    clip_model,
):
    states_t   = torch.stack(buffer._states).to(device)
    canvases_cpu = buffer._canvases
    acts_t     = torch.stack(buffer._acts).to(device)
    old_lp_t   = torch.stack(buffer._lps).to(device)

    advs, rets = compute_gae(buffer._rews, buffer._vals, buffer._dones, last_val)
    advs_n     = ((advs - advs.mean()) / (advs.std() + 1e-8)).to(device)
    rets       = rets.to(device)
    T          = len(buffer._rews)

    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(T)
        for start in range(0, T, MINI_BATCH):
            mb = idx[start : start + MINI_BATCH]
            if len(mb) < 4:
                continue

            mb_canvases = torch.stack(
                [
                    canvases_cpu[i.item()].to(device=device, dtype=torch.float32)
                    for i in mb
                ]
            )

            new_lp, new_val, entropy = policy.evaluate(
                states_t[mb], mb_canvases, acts_t[mb]
            )
            ratio  = (new_lp - old_lp_t[mb]).exp()
            surr1  = ratio * advs_n[mb]
            surr2  = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * advs_n[mb]
            a_loss = -torch.min(surr1, surr2).mean()
            c_loss = F.mse_loss(new_val, rets[mb])
            e_loss = -entropy.mean()
            ppo_loss = a_loss + VALUE_COEF * c_loss + entropy_coef * e_loss

            mb_loss = _model_based_loss(
                policy,
                states_t,
                canvases_cpu,
                mb,
                device,
                renderer,
                text_embed,
                sketch_embed,
                clip_model,
            )

            loss = ppo_loss + MODEL_BASED_W * mb_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
            optimizer.step()


# ══════════════════════════════════════════════════════════════════════════════
#  Training
# ══════════════════════════════════════════════════════════════════════════════


def train(
    prompt, n_episodes, save_dir, device, renderer_path="renderer.pkl", sketch_path=None
):
    os.makedirs(save_dir, exist_ok=True)

    print("[INFO] Loading CLIP ViT-B/32 ...")
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad_(False)

    print("[INFO] Loading neural renderer ...")
    renderer = NeuralRenderer(renderer_path, device)

    with torch.no_grad():
        tokens = clip.tokenize([prompt]).to(device)
        t_emb  = clip_model.encode_text(tokens).float()
        t_emb  = t_emb / t_emb.norm(dim=-1, keepdim=True)

    sketch_emb = load_sketch_embed(sketch_path, clip_model, device)

    policy    = ActorCritic().to(device)
    optimizer = optim.Adam(policy.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_episodes, eta_min=LR_MIN
    )
    buffer = RolloutBuffer()

    sim_window   = deque(maxlen=100)
    cov_window   = deque(maxlen=100)
    alpha_window = deque(maxlen=100)
    rew_window   = deque(maxlen=100)
    all_sims, all_rews = [], []
    best_sim       = -float("inf")
    best_canvas_np = None
    episodes_since_best = 0

    env           = PaintEnv(t_emb, sketch_emb, clip_model, renderer, device)
    state, canvas = env.reset()

    ep_num    = 0
    ep_reward = 0.0
    t_start   = time.time()

    print(f"[INFO] prompt='{prompt}'  episodes={n_episodes}  device={device}")
    print(
        f"[INFO] sketch={'yes (' + sketch_path + ')' if sketch_path else 'no (zeros)'}"
    )
    print(f"[INFO] STATE_DIM={STATE_DIM}  ACTION_DIM={ACTION_DIM}")
    print(
        f"[INFO] r0/r1 init bias: sigmoid({_BIAS_R}) = {torch.sigmoid(torch.tensor(_BIAS_R)):.4f}"
    )
    print(
        f"[INFO] Stroke size penalty: {STROKE_SIZE_PENALTY_MIN} → {STROKE_SIZE_PENALTY_MAX} "
        f"(linear ramp over {N_STROKE_BUNDLES} steps)"
    )
    print(
        f"[INFO] Center reward: {'ENABLED  (w={CENTER_REWARD_W}, sigma={CENTER_REWARD_SIGMA})' if CENTER_REWARD_ENABLED else 'disabled'}"
    )
    print("-" * 72)

    while ep_num < n_episodes:
        progress     = ep_num / max(n_episodes - 1, 1)
        entropy_coef = ENTROPY_START + progress * (ENTROPY_END - ENTROPY_START)

        canvas_before_cpu = env.get_canvas_snapshot()

        policy.eval()
        with torch.no_grad():
            action, log_p, val = policy.act(state.to(device), canvas.to(device))

        next_state, next_canvas, reward, done, info = env.step(action)
        buffer.push(state, canvas_before_cpu, action, reward, log_p, val, done)
        ep_reward += reward
        state  = next_state
        canvas = next_canvas

        if done:
            ep_num              += 1
            episodes_since_best += 1
            sim = info["clip_sim"]
            cov = info["coverage"]
            alpha_window.append(info["mean_alpha"])
            sim_window.append(sim)
            cov_window.append(cov)
            rew_window.append(ep_reward)
            all_sims.append(sim)
            all_rews.append(reward)

            canvas_np = canvas.cpu().numpy().transpose(1, 2, 0)
            if sim > best_sim:
                best_sim       = sim
                best_canvas_np = canvas_np.copy()
                _save_canvas(best_canvas_np, os.path.join(save_dir, "best.png"))
                episodes_since_best = 0

            if ep_num % LOG_EVERY == 0:
                elapsed = time.time() - t_start
                print(
                    f"  ep {ep_num:5d}/{n_episodes} | "
                    f"sim={np.mean(sim_window):.4f} | "
                    f"cov={np.mean(cov_window):.3f} | "
                    f"alpha={np.mean(alpha_window):.3f} | "
                    f"rew={np.mean(rew_window):+.3f} | "
                    f"best={best_sim:.4f} | "
                    f"ent={entropy_coef:.3f} | "
                    f"pat={episodes_since_best:.0f} | "
                    f"t={elapsed:.0f}s"
                )
                _save_canvas(
                    canvas_np, os.path.join(save_dir, f"canvas_ep{ep_num:05d}.png")
                )
                scheduler.step()

            if episodes_since_best > PATIENCE:
                print(f"[EVAL]: No improvement since {PATIENCE} episodes. Early Stopping.")
                break

            state, canvas = env.reset()
            ep_reward     = 0.0

        if len(buffer) >= UPDATE_EVERY:
            policy.train()
            with torch.no_grad():
                _, _, last_val = policy.act(state.to(device), canvas.to(device))
            ppo_update(
                policy,
                optimizer,
                buffer,
                last_val,
                device,
                entropy_coef,
                renderer,
                t_emb,
                sketch_emb,
                clip_model,
            )
            buffer.clear()

    print("-" * 72)
    print(f"[DONE] Best CLIP sim = {best_sim:.4f}")

    torch.save(policy.state_dict(), os.path.join(save_dir, "policy.pt"))
    if best_canvas_np is not None:
        Image.fromarray((best_canvas_np * 255).astype(np.uint8)).resize(
            (256, 256), Image.NEAREST
        ).save(os.path.join(save_dir, "best_256.png"))

    _plot_curves(all_sims, all_rews, prompt, save_dir)
    return best_canvas_np, best_sim


# ══════════════════════════════════════════════════════════════════════════════
#  Evaluation
# ══════════════════════════════════════════════════════════════════════════════


def evaluate(prompt, checkpoint, renderer_path, save_dir, device, sketch_path=None):
    os.makedirs(save_dir, exist_ok=True)
    clip_model, _ = clip.load("ViT-B/32", device=device)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad_(False)
    with torch.no_grad():
        tokens = clip.tokenize([prompt]).to(device)
        t_emb  = clip_model.encode_text(tokens).float()
        t_emb  = t_emb / t_emb.norm(dim=-1, keepdim=True)

    sketch_emb = load_sketch_embed(sketch_path, clip_model, device)
    renderer   = NeuralRenderer(renderer_path, device)
    policy     = ActorCritic().to(device)
    policy.load_state_dict(
        torch.load(checkpoint, map_location=device, weights_only=False)
    )
    policy.eval()

    env           = PaintEnv(t_emb, sketch_emb, clip_model, renderer, device)
    state, canvas = env.reset()
    frames        = [canvas.cpu().numpy().transpose(1, 2, 0)]

    with torch.no_grad():
        for step in range(N_STROKE_BUNDLES):
            action, _, _ = policy.act(state.to(device), canvas.to(device))
            state, canvas, _, done, info = env.step(action)
            frames.append(canvas.cpu().numpy().transpose(1, 2, 0))
            print(
                f"  bundle {step+1:2d}/{N_STROKE_BUNDLES}  "
                f"sim={info['clip_sim']:.4f}  "
                f"cov={info['coverage']:.3f}  "
                f"alpha={info['mean_alpha']:.3f}  "
                f"penalty_w={info['penalty_w']:.1f}"
            )

    gif_path = os.path.join(save_dir, "drawing.gif")
    pils     = [
        Image.fromarray((f * 255).astype(np.uint8)).resize((256, 256), Image.NEAREST)
        for f in frames
    ]
    pils[0].save(gif_path, save_all=True, append_images=pils[1:], duration=300, loop=0)
    print(f"[EVAL] GIF -> {gif_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  Utilities
# ══════════════════════════════════════════════════════════════════════════════


def _save_canvas(canvas_np, path):
    Image.fromarray((canvas_np * 255).astype(np.uint8)).save(path)


def _plot_curves(sims, rews, prompt, save_dir):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    win = min(50, len(sims))
    ax1.plot(sims, alpha=0.3, color="steelblue", linewidth=0.7, label="sim")
    if len(sims) >= win:
        sm = np.convolve(sims, np.ones(win) / win, mode="valid")
        ax1.plot(
            range(win - 1, len(sims)),
            sm,
            color="steelblue",
            linewidth=2,
            label=f"{win}-ep avg",
        )
    ax1.set_ylabel("CLIP similarity")
    ax1.set_title(f"Training — '{prompt}'")
    ax1.legend(fontsize=8)
    ax2.plot(rews, alpha=0.3, color="darkorange", linewidth=0.7, label="reward")
    if len(rews) >= win:
        cm = np.convolve(rews, np.ones(win) / win, mode="valid")
        ax2.plot(
            range(win - 1, len(rews)),
            cm,
            color="darkorange",
            linewidth=2,
            label=f"{win}-ep avg",
        )
    ax2.set_ylabel("Reward")
    ax2.set_xlabel("Episode")
    ax2.legend(fontsize=8)
    plt.tight_layout()
    out = os.path.join(save_dir, "training_curve.png")
    plt.savefig(out, dpi=120)
    plt.close()
    print(f"[DONE] Training curve -> {out}")


# ══════════════════════════════════════════════════════════════════════════════
#  Entry point
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--prompt",
        type=str,
        default="A painting of a red apple with a small leaf",
    )
    parser.add_argument("--episodes", type=int, default=5000)
    parser.add_argument("--save_dir", type=str, default="outputs")
    parser.add_argument("--renderer", type=str, default=f"{BASE_URL}/renderer.pkl")
    parser.add_argument(
        "--sketch",
        type=str,
        default=None,
        help="Path to a reference sketch PNG (optional)",
    )
    parser.add_argument("--eval", action="store_false")
    parser.add_argument(
        "--device", type=str, default="auto", choices=["auto", "cpu", "cuda", "mps"]
    )
    args, _ = parser.parse_known_args()

    if args.device == "auto":
        if torch.cuda.is_available():
            dev = torch.device("cuda")
        elif torch.backends.mps.is_available():
            dev = torch.device("mps")
        else:
            dev = torch.device("cpu")
    else:
        dev = torch.device(args.device)

    print(f"[INFO] Device: {dev}")

    if args.eval:
        ckpt = os.path.join(args.save_dir, "policy.pt")
        if not os.path.isfile(ckpt):
            raise FileNotFoundError(f"No checkpoint at {ckpt}")
        evaluate(args.prompt, ckpt, args.renderer, args.save_dir, dev, args.sketch)
    else:
        train(
            args.prompt, args.episodes, args.save_dir, dev, args.renderer, args.sketch
        )

[INFO] Device: cuda
[INFO] Loading CLIP ViT-B/32 ...
[INFO] Loading neural renderer ...
[NeuralRenderer] '/content/drive/MyDrive/RL Project/renderer.pkl'  in=(10,) → alpha=(128×128)
[sketch] No sketch provided — sketch embedding zeroed.
[INFO] prompt='A painting of a sunflower'  episodes=5000  device=cuda
[INFO] sketch=no (zeros)
[INFO] STATE_DIM=1793  ACTION_DIM=65
[INFO] r0/r1 init bias: sigmoid(0.0) = 0.5000
[INFO] Stroke size penalty: 0.1 → 0.1 (linear ramp over 5 steps)
[INFO] Center reward: disabled
------------------------------------------------------------------------
  ep    20/5000 | sim=0.2002 | cov=0.310 | alpha=0.047 | rew=+4.037 | best=0.2194 | ent=0.020 | pat=12 | t=2s


/tmp/ipykernel_5827/1029143613.py:942: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  ep    40/5000 | sim=0.2016 | cov=0.320 | alpha=0.045 | rew=+4.237 | best=0.2300 | ent=0.020 | pat=0 | t=5s
  ep    60/5000 | sim=0.2023 | cov=0.323 | alpha=0.045 | rew=+4.335 | best=0.2300 | ent=0.020 | pat=20 | t=8s
  ep    80/5000 | sim=0.2020 | cov=0.326 | alpha=0.045 | rew=+4.287 | best=0.2300 | ent=0.020 | pat=40 | t=10s
  ep   100/5000 | sim=0.2018 | cov=0.330 | alpha=0.045 | rew=+4.257 | best=0.2300 | ent=0.020 | pat=60 | t=12s
  ep   120/5000 | sim=0.2027 | cov=0.334 | alpha=0.044 | rew=+4.389 | best=0.2300 | ent=0.020 | pat=80 | t=15s
  ep   140/5000 | sim=0.2027 | cov=0.335 | alpha=0.044 | rew=+4.393 | best=0.2300 | ent=0.019 | pat=100 | t=18s
  ep   160/5000 | sim=0.2032 | cov=0.336 | alpha=0.043 | rew=+4.454 | best=0.2300 | ent=0.019 | pat=120 | t=20s
  ep   180/5000 | sim=0.2046 | cov=0.336 | alpha=0.042 | rew=+4.662 | best=0.2335 | ent=0.019 | pat=18 | t=23s
  ep   200/5000 | sim=0.2059 | cov=0.338 | alpha=0.040 | rew=+4.834 | best=0.2335 | ent=0.019 | pat=38 | t=25s
  

In [ ]:
!zip -r outputs.zip outputs/

updating: outputs/ (stored 0%)
updating: outputs/canvas_ep04020.png (stored 0%)
updating: outputs/canvas_ep03480.png (stored 0%)
updating: outputs/canvas_ep02380.png (stored 0%)
updating: outputs/canvas_ep08420.png (stored 0%)
updating: outputs/canvas_ep00480.png (stored 0%)
updating: outputs/canvas_ep00020.png (deflated 1%)
updating: outputs/canvas_ep06860.png (stored 0%)
updating: outputs/canvas_ep01540.png (stored 0%)
updating: outputs/canvas_ep03120.png (stored 0%)
updating: outputs/canvas_ep07280.png (stored 0%)
updating: outputs/canvas_ep00420.png (stored 0%)
updating: outputs/canvas_ep03900.png (stored 0%)
updating: outputs/canvas_ep03940.png (stored 0%)
updating: outputs/canvas_ep06480.png (stored 0%)
updating: outputs/canvas_ep08020.png (stored 0%)
updating: outputs/canvas_ep00580.png (stored 0%)
updating: outputs/canvas_ep02020.png (stored 0%)
updating: outputs/canvas_ep00740.png (stored 0%)
updating: outputs/canvas_ep00200.png (deflated 0%)
updating: outputs/canvas_ep00160.p

In [32]:
!zip outputs.zip outputs/drawing.gif outputs/training_curve.png outputs/best.png outputs/best_256.png

updating: outputs/drawing.gif (deflated 1%)
updating: outputs/training_curve.png (deflated 3%)
updating: outputs/best.png (stored 0%)
updating: outputs/best_256.png (deflated 2%)


In [ ]:
!rm -rf outputs/